# Hands-on Task 1: Extend the basic Pipeline (No LLM)

## Objective
Extend an existing LangGraph workflow by introducing a new node called 'word_count'.

### Worlflow
START > count > word_count > summarise > END

### Features
- Count characters
- Count words
- Generate final summary containing both statistics

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define state structure
class TextState(TypedDict):
    text: str
    char_count: int
    word_count: int
    summary: str


# Count characters
def count_characters(state: TextState):
    return {
        "char_count": len(state["text"])
    }


# Count words
def count_words(state: TextState):
    return {
        "word_count": len(state["text"].split())
    }


# Create summary
def make_summary(state: TextState):
    return {
        "summary": (
            f"Summary: '{state['text']}' — "
            f"{state['char_count']} characters, "
            f"{state['word_count']} words."
        )
    }


# Build Graph
graph_builder = StateGraph(TextState)

graph_builder.add_node("count", count_characters)
graph_builder.add_node("word_count", count_words)
graph_builder.add_node("summarise", make_summary)

graph_builder.add_edge(START, "count")
graph_builder.add_edge("count", "word_count")
graph_builder.add_edge("word_count", "summarise")
graph_builder.add_edge("summarise", END)

graph = graph_builder.compile()


# Test
result = graph.invoke({
    "text": "LangGraph makes stateful AI workflows easy!"
})

print(result["summary"])

Summary: 'LangGraph makes stateful AI workflows easy!' — 43 characters, 6 words.


# Hands-on Task 2: Review Pipeline using Claude

## Objective
Create a multi-step LangGraph workflow that:

1. Generates a product review
2. Detects sentiment
3. Generates a brand response

### Flow

START → review_node → sentiment_node → reply_node → END

In [22]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
load_dotenv()

import os

# Claude Model
llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0.5
)

# State Definition
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

In [23]:
# Step 1 - Generate Review
def review_node(state: ReviewState):

    prompt = f"""
    Write a short customer review (2-3 sentences)
    for the following product:

    Product: {state['product']}
    """

    review = llm.invoke(prompt).content

    return {
        "review": review
    }


# Step 2 - Sentiment Classification
def sentiment_node(state: ReviewState):

    prompt = f"""
    Classify the sentiment of this review.

    Review:
    {state['review']}

    Return ONLY one word:
    Positive
    Negative
    Neutral
    """

    sentiment = llm.invoke(prompt).content.strip()

    return {
        "sentiment": sentiment
    }


# Step 3 - Brand Reply
def reply_node(state: ReviewState):

    prompt = f"""
    A customer left a {state['sentiment']} review.

    Review:
    {state['review']}

    Write a one-line professional brand response.
    """

    reply = llm.invoke(prompt).content

    return {
        "reply": reply
    }

In [24]:
# Build Graph

builder = StateGraph(ReviewState)

builder.add_node("review_node", review_node)
builder.add_node("sentiment_node", sentiment_node)
builder.add_node("reply_node", reply_node)

builder.add_edge(START, "review_node")
builder.add_edge("review_node", "sentiment_node")
builder.add_edge("sentiment_node", "reply_node")
builder.add_edge("reply_node", END)

graph = builder.compile()


In [25]:
# Test Input

result = graph.invoke({
    "product": "wireless noise-cancelling headphones"
})

print("Product:")
print(result["product"])

print("\nGenerated Review:")
print(result["review"])

print("\nSentiment:")
print(result["sentiment"])

print("\nBrand Reply:")
print(result["reply"])

Product:
wireless noise-cancelling headphones

Generated Review:
# Customer Review: Wireless Noise-Cancelling Headphones

These headphones are a game-changer for my daily commute and work-from-home setup! The noise cancellation is incredibly effective, blocking out everything from traffic to office chatter, and the sound quality is crisp and balanced across all genres. Battery life easily gets me through a full week of regular use, and the comfortable fit means I can wear them for hours without any fatigue.

Sentiment:
Positive

Brand Reply:
# Brand Response:

Thank you so much for the wonderful review—we're thrilled that our headphones are enhancing your daily experience, and we truly appreciate your loyalty!


# Hands-on Task 3: Three-Way Conditional Router

## Workflow

```text
                START
                   |
                   v
              +---------+
              | Router  |
              +---------+
                   |
                   v
              +----------+
              | Decision |
              +----------+
               /    |    \
              /     |     \
             v      v      v

     +-------------+  +-------------+  +-------------+
     | science_node|  | history_node|  | general_node|
     +-------------+  +-------------+  +-------------+
             \            |             /
              \           |            /
               \          |           /
                v         v          v

                      +------+
                      | END  |
                      +------+
```

In [26]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0.3
)

# State Structure
class QuestionState(TypedDict):
    question: str
    answer: str

In [27]:
# Router Function

def router(
    state: QuestionState
) -> Literal["science_node", "history_node", "general_node"]:

    question = state["question"].lower()

    science_keywords = [
        "physics",
        "chemistry",
        "biology",
        "science",
        "atom",
        "energy",
        "gravity",
        "planet"
    ]

    history_keywords = [
        "history",
        "war",
        "king",
        "empire",
        "ancient",
        "independence",
        "revolution"
    ]

    if any(word in question for word in science_keywords):
        return "science_node"

    elif any(word in question for word in history_keywords):
        return "history_node"

    return "general_node"

In [28]:
# Science Teacher Persona

def science_node(state: QuestionState):

    prompt = f"""
    You are a science teacher.

    Explain clearly and simply:

    {state['question']}
    """

    answer = llm.invoke(prompt).content

    return {"answer": answer}


# Historian Persona

def history_node(state: QuestionState):

    prompt = f"""
    You are a historian.

    Provide a historical explanation:

    {state['question']}
    """

    answer = llm.invoke(prompt).content

    return {"answer": answer}


# General Assistant Persona

def general_node(state: QuestionState):

    prompt = f"""
    You are a helpful assistant.

    Answer the question:

    {state['question']}
    """

    answer = llm.invoke(prompt).content

    return {"answer": answer}

In [29]:
# Build Graph

builder = StateGraph(QuestionState)

builder.add_node("science_node", science_node)
builder.add_node("history_node", history_node)
builder.add_node("general_node", general_node)

builder.add_conditional_edges(
    START,
    router
)

builder.add_edge("science_node", END)
builder.add_edge("history_node", END)
builder.add_edge("general_node", END)

graph = builder.compile()

In [30]:
# Test Questions

questions = [
    "What is gravity?",
    "Why did the Roman Empire fall?",
    "How does photosynthesis work?",
    "How can I improve my productivity?"
]

for q in questions:

    result = graph.invoke({
        "question": q
    })

    print("=" * 80)
    print("Question:", q)
    print("\nAnswer:")
    print(result["answer"])
    print()

Question: What is gravity?

Answer:
# What is Gravity?

Gravity is a **force that pulls objects toward each other**. It's one of the fundamental forces in the universe.

## The Simple Version

Imagine you drop a ball. It falls down, right? That's gravity pulling it toward Earth. Everything with mass (weight) has gravity, and it pulls on everything else.

## Key Points

**What it does:**
- Pulls objects downward (toward Earth's center)
- Keeps planets orbiting the Sun
- Keeps the Moon orbiting Earth
- Holds galaxies together

**Why it happens:**
- Every object with mass creates a gravitational pull
- The bigger the mass, the stronger the pull
- The closer objects are, the stronger the pull

## Real-Life Examples

- 🍎 An apple falls from a tree
- 🌍 You stay on the ground instead of floating away
- 🌙 The Moon doesn't fly off into space
- ⭐ Planets orbit the Sun

## The Bottom Line

Gravity is the invisible force that keeps us on Earth and holds the universe together. Without it, everythin